# 03 · Sensibilité aux seuils

Chaque seuil est balayé seul, les autres restant à leur valeur par défaut. Le rejeu est chargé une seule fois ;
le moteur est relancé pour chaque valeur (`analyser.evaluation.sweep`). Les scores portent sur tous les épisodes.
La ligne pointillée marque la valeur par défaut.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent  # AWS/Analyser
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))  # repli si iCloud masque le .venv (voir README)

import matplotlib.pyplot as plt
import pandas as pd

import _viz as viz
from analyser import generator
from analyser.config import Thresholds
from analyser.evaluation import analyse, episodes_from_ground_truth, evaluate, load_replay, sweep
from analyser.replay_files import read_ground_truth

viz.apply_style()
pd.set_option("display.precision", 2)

REPLAY = ROOT / "data" / "replay" / "ref"
if not (REPLAY / "batches.jsonl").exists():  # jeu de référence : 10 machines x 8 h, seed 42
    generator.write_replay(generator.generate(devices=10, hours=8, seed=42), REPLAY)

fetched = load_replay(REPLAY / "batches.jsonl")
ground_truth = read_ground_truth(REPLAY / "ground_truth.csv")
episodes = episodes_from_ground_truth(ground_truth, data_end_ms=fetched.now_ms)
eps = viz.episodes_frame(episodes)
print(f"{len(fetched.readings):,} lectures, {len(ground_truth):,} événements de vérité terrain, {len(episodes)} épisodes")

In [ ]:
DEFAULTS = Thresholds()


def run(parameter, values, scenario, label):
    frame = viz.sweep_frame(sweep(fetched, episodes, DEFAULTS, parameter, values), parameter, scenario)
    viz.plot_sweep(frame, parameter, label, getattr(DEFAULTS, parameter))
    plt.show()
    return frame[[parameter, "recall", "precision", "alerts", "false_alerts", "median_latency_s"]]

## 1. Emballement thermique : écart à la consigne

En dessous du bruit de mesure (±0,3 °C), chaque machine au repos déclenche des alertes.

In [ ]:
run("thermal_delta_c", [0.1, 0.2, 0.3, 0.5, 1, 2, 4, 6, 8, 10, 12, 14, 16], "thermal_runaway", "consigne + Δ (°C)")

## 2. Emballement thermique : lectures consécutives

In [ ]:
run("thermal_consecutive", [1, 2, 3, 4, 5, 6, 8, 10], "thermal_runaway", "lectures consécutives au-dessus du seuil")

## 3. Fuite mémoire : baisse minimale du heap après une mise à jour

In [ ]:
run("leak_min_drop_bytes", [50, 100, 200, 500, 1000, 2000, 4000, 6000, 8000, 9000], "memory_leak_fota", "baisse minimale (octets)")

## 4. Fuite mémoire : nombre de mises à jour consécutives avec baisse

In [ ]:
run("leak_streak", [1, 2, 3, 4, 5, 6], "memory_leak_fota", "mises à jour consécutives")

## 5. Machine muette : durée minimale du trou

In [ ]:
run("silent_gap_s", [3, 5, 10, 20, 30, 45, 60, 90, 120], "silent_device", "trou minimal (s)")

## Conclusions

- **`thermal_delta_c = 4 °C`, `thermal_consecutive = 3`** : aucune fausse alerte, WARNING ≈ 10 s après le début de
  l'emballement, soit ≈ 14 s avant le seuil critique. Sous 0,3 °C (le bruit de mesure), les fausses alertes explosent ;
  à partir de ≈ 14 °C, l'alerte précoce arrive en même temps que le CRITICAL et n'apporte plus rien.
- **`leak_min_drop_bytes = 2000`** : tout seuil entre ≈ 50 et 6 000 octets donne le même résultat sur ces données ; 2 000
  laisse une marge confortable au-dessus du bruit du heap (±200 octets) et en dessous d'une fuite (8 000).
- **`leak_streak = 2`** : avec `1`, la fuite serait signalée ≈ 2 min après la première mise à jour fautive (médiane 24 min),
  sans fausse alerte, car le simulateur ne fait jamais baisser le heap autrement. Sur de vraies machines, une seule baisse
  peut être légitime (nouveau firmware plus gourmand) : exiger deux mises à jour consécutives coûte un cycle (médiane 77 min)
  mais reste des heures avant le crash. À partir de 3, des fuites ne sont plus signalées à temps ; à partir de 5, plus aucune.
- **`silent_gap_s = 10 s`** (5 lectures manquées) : tous les silences simulés (30 à 120 s) sont détectés, sans fausse alerte.
  Au-delà de 30 s, les silences les plus courts échappent au détecteur.